In [ ]:
import argparse
from pathlib import Path
import dask.config
import numpy as np
import pyarrow.parquet as pq
import pyarrow.dataset as ds
import pyarrow as pa
import pandas as pd
import dask.dataframe as dd
from dask.distributed import Client
from tqdm import tqdm
from pathlib import Path

from cbookcore import BookCore

dask.config.set({
    "temporary_directory": r"E:\dask_disk_spilling",
    "distributed.worker.memory.target": 0.6,
    "distributed.worker.memory.spill": 0.7,
    "distributed.worker.memory.pause": 0.99,
    "distributed.worker.memory.terminate": 0.999,
    })
client = Client()

In [2]:
books_dir = Path(r"E:\tmp\OKX-BL10-BTC-USDT")
trades_dir = Path(r"E:\tmp\OKX-Trades-BTC-USDT")
books_pfs = list(books_dir.glob("*.parquet"))
trades_pfs = list(trades_dir.glob("*.parquet"))

In [3]:
from DataFile.data_name import DataName

tss = []
for i, pf in enumerate(books_pfs):
    t = pf.stem.split('-')
    if tss and not (tss[-1] < t[-2]):
        print(i,pf)
        break
    tss.extend([t[-2], t[-1]])
else:
    print('Sorted.')
# all(a < b for a, b in zip(tss, tss[1:]))
    

Sorted.


In [4]:
books_df = dd.read_parquet(books_pfs)
trades_df = dd.read_parquet(trades_pfs)

In [8]:
books_df['timestamp'] = dd.to_datetime(books_df['ts'], unit='ms')
books_df['book_timestamp_back'] = dd.to_datetime(books_df['ts'], unit='ms')
trades_df['timestamp'] = dd.to_datetime(trades_df['ts'], unit='ms')
books_df = books_df.rename(columns={'timestamp': 'books_timestamp'})
trades_df = trades_df.set_index('timestamp', sorted=True)
books_df = books_df.set_index('books_timestamp', sorted=True)

In [10]:
merged_df = dd.merge_asof(
    left=trades_df,
    right=books_df,
    left_index=True,
    right_index=True,
    direction='backward'
)
merged_df = merged_df.drop(['arg','ts_x','ts_y'], axis=1)
merged_df = merged_df.dropna()

In [14]:
merged_df['time_diff_seconds'] = (merged_df['timestamp'] - merged_df['book_timestamp_back']).dt.total_seconds()

In [15]:
print(merged_df[['book_timestamp_back']].head(3))

                            book_timestamp_back
timestamp                                      
2025-02-28 16:01:49.560 2025-02-28 16:01:49.503
2025-02-28 16:01:49.653 2025-02-28 16:01:49.603
2025-02-28 16:01:50.036 2025-02-28 16:01:50.003


In [19]:
print(merged_df[['time_diff_seconds']].max().compute())

KilledWorker: Attempted to run task ('resolveoverlappingdivisions-f572350559b4b7643402a136a59d5513', 11) on 4 different workers, but all those workers died while running it. The last worker that attempt to run the task was tcp://127.0.0.1:55606. Inspecting worker logs is often a good next step to diagnose what went wrong. For more information see https://distributed.dask.org/en/stable/killed.html.